In [0]:
%pip install -U \
    "mcp>=1.9" \
    "databricks-mcp" \
    "databricks-sdk[openai]" \
    "mlflow>=3.1.0" \
    "databricks-agents>=1.0.0"

In [0]:
dbutils.library.restartPython()

In [0]:
import mcp
import databricks_mcp
import mlflow
import databricks.sdk

print("MCP: imported successfully ✅")
print("databricks_mcp: imported successfully ✅")
print("MLflow:", mlflow.__version__)
print("Databricks SDK:", databricks.sdk.__version__)

Databricks specifically provides DatabricksMCPClient for connecting custom agent code to an MCP Service, and the same client works with external MCP Services.

Create the MCP Client

In [0]:
from databricks_mcp import DatabricksMCPClient
from databricks.sdk import WorkspaceClient

# Create Databricks workspace client
workspace_client = WorkspaceClient()

# Get current workspace URL automatically
host = workspace_client.config.host

print("Workspace:", host)

# Our MCP Service
mcp_server_url = (
    f"{host}/ai-gateway/mcp-services/"
    "workspace.default.github_mcp_service"
)

print("MCP Service URL:", mcp_server_url)

# Create MCP client
mcp_client = DatabricksMCPClient(
    server_url=mcp_server_url,
    workspace_client=workspace_client
)

print("MCP Client created successfully ✅")

The important thing is that we are NOT directly connecting the agent to GitHub. The agent will eventually use the Databricks MCP Service, which provides the governed MCP tool layer. Databricks documents this DatabricksMCPClient pattern for MCP Services.

In [0]:
tools = mcp_client.list_tools()

print("Number of tools:", len(tools))
print("\nAvailable tools:")

for tool in tools:
    print("-", tool.name)

In [0]:
import nest_asyncio

nest_asyncio.apply()

Call a GitHub MCP tool directly

In [0]:
result = mcp_client.call_tool(
    "get_me",
    {}
)

print(result.content)

Until now, we manually called:

In [0]:
tool = tools[0]

print("Tool object:")
print(tool)

print("\nTool name:")
print(tool.name)

print("\nTool description:")
print(tool.description)

In [0]:
from mlflow.pyfunc import ResponsesAgent

print("ResponsesAgent imported successfully ✅")
print(ResponsesAgent)

In [0]:
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()

endpoints = ws.serving_endpoints.list()

for ep in endpoints:
    print(ep.name)

In [0]:
%pip install -U databricks-openai

In [0]:
# ============================================================
# GITHUB MCP + RESPONSES AGENT
# Databricks MCP Service → GitHub MCP → GitHub
# ============================================================

import json
import uuid

import mlflow

from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
)

from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient
from databricks_openai import DatabricksOpenAI


# ============================================================
# 1. CONFIGURATION
# ============================================================

LLM_ENDPOINT_NAME = "databricks-gpt-oss-120b"

MCP_SERVICE_NAME = (
    "workspace.default.github_mcp_service"
)

SYSTEM_PROMPT = """
You are a helpful GitHub assistant.

You have access to GitHub through MCP tools.

Use the available GitHub MCP tools whenever the
user asks for information or an action involving GitHub.

Do not invent GitHub information.

If a tool is unavailable or fails, clearly explain that.
"""


# ============================================================
# 2. DATABRICKS WORKSPACE CLIENT
# ============================================================

workspace_client = WorkspaceClient()

host = workspace_client.config.host

print("Workspace:")
print(host)


# ============================================================
# 3. MCP SERVICE URL
# ============================================================

mcp_server_url = (
    f"{host}/ai-gateway/mcp-services/"
    f"{MCP_SERVICE_NAME}"
)

print("\nMCP Service:")
print(mcp_server_url)


# ============================================================
# 4. CREATE MCP CLIENT
# ============================================================

mcp_client = DatabricksMCPClient(
    server_url=mcp_server_url,
    workspace_client=workspace_client,
)

print("\nMCP Client created ✅")


# ============================================================
# 5. DISCOVER MCP TOOLS
# ============================================================

mcp_tools = mcp_client.list_tools()

print(
    f"\nDiscovered MCP tools: {len(mcp_tools)}"
)

for tool in mcp_tools:
    print("-", tool.name)


# ============================================================
# 6. CONVERT MCP TOOLS TO LLM TOOL DEFINITIONS
# ============================================================

def convert_mcp_tool_to_openai_tool(tool):

    input_schema = dict(
        tool.input_schema
    )

    if "properties" not in input_schema:
        input_schema["properties"] = {}

    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "",
            "parameters": input_schema,
        },
    }


llm_tools = [
    convert_mcp_tool_to_openai_tool(tool)
    for tool in mcp_tools
]

print(
    f"\nLLM tools prepared: {len(llm_tools)}"
)


# ============================================================
# 7. TOOL LOOKUP
# ============================================================

tool_map = {
    tool.name: tool
    for tool in mcp_tools
}


# ============================================================
# 8. CREATE THE RESPONSES AGENT
# ============================================================

class GitHubMCPAgent(ResponsesAgent):

    # --------------------------------------------------------
    # LLM CALL
    # --------------------------------------------------------

    def call_llm(
        self,
        messages,
        include_tools=True,
    ):

        client = DatabricksOpenAI()

        tools = (
            llm_tools
            if include_tools
            else None
        )

        response = client.chat.completions.create(
            model=LLM_ENDPOINT_NAME,
            messages=messages,
            tools=tools,
        )

        return response


    # --------------------------------------------------------
    # MCP RESULT → TEXT
    # --------------------------------------------------------

    def extract_mcp_result(
        self,
        result,
    ):

        parts = []

        for content in result.content:

            if hasattr(content, "text"):
                parts.append(content.text)

            else:
                parts.append(str(content))

        return "\n".join(parts)


    # --------------------------------------------------------
    # AGENT PREDICT
    # --------------------------------------------------------

    def predict(
        self,
        request: ResponsesAgentRequest,
    ) -> ResponsesAgentResponse:

        # ====================================================
        # BUILD MESSAGE HISTORY
        # ====================================================

        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            }
        ]

        for item in request.input:

            item_dict = item.model_dump(
                exclude_none=True
            )

            # User message
            if item_dict.get("role") == "user":

                messages.append(
                    {
                        "role": "user",
                        "content": item_dict.get(
                            "content",
                            "",
                        ),
                    }
                )

            # Assistant message
            elif item_dict.get("role") == "assistant":

                messages.append(
                    {
                        "role": "assistant",
                        "content": item_dict.get(
                            "content",
                            "",
                        ),
                    }
                )


        # ====================================================
        # FIRST LLM CALL
        # ====================================================

        print("\n🤖 Calling LLM...")

        response = self.call_llm(
            messages,
            include_tools=True,
        )

        assistant_message = (
            response.choices[0].message
        )

        tool_calls = (
            assistant_message.tool_calls
            or []
        )


        # ====================================================
        # NO TOOL REQUIRED
        # ====================================================

        if not tool_calls:

            answer = (
                assistant_message.content
                or ""
            )

            return ResponsesAgentResponse(
                output=[
                    self.create_text_output_item(
                        text=answer,
                        id=uuid.uuid4().hex,
                    )
                ],
                custom_outputs=(
                    request.custom_inputs
                ),
            )


        # ====================================================
        # LLM REQUESTED MCP TOOL(S)
        # ====================================================

        print(
            f"\n🔧 LLM requested "
            f"{len(tool_calls)} tool(s)"
        )


        # Add assistant's tool-call message
        messages.append(
            {
                "role": "assistant",
                "content": (
                    assistant_message.content
                    or None
                ),
                "tool_calls": [
                    {
                        "id": call.id,
                        "type": "function",
                        "function": {
                            "name": (
                                call.function.name
                            ),
                            "arguments": (
                                call.function.arguments
                            ),
                        },
                    }
                    for call in tool_calls
                ],
            }
        )


        # ====================================================
        # EXECUTE MCP TOOLS
        # ====================================================

        for call in tool_calls:

            tool_name = (
                call.function.name
            )

            arguments = json.loads(
                call.function.arguments
            )

            print(
                f"\n🔧 MCP Tool:"
                f" {tool_name}"
            )

            print(
                f"Arguments:"
                f" {arguments}"
            )


            # ----------------------------------------------
            # Check tool
            # ----------------------------------------------

            if tool_name not in tool_map:

                tool_result = (
                    f"Tool '{tool_name}' "
                    f"is not available."
                )

            else:

                try:

                    # ------------------------------------------
                    # ACTUAL MCP CALL
                    # ------------------------------------------

                    result = (
                        mcp_client.call_tool(
                            tool_name,
                            arguments,
                        )
                    )

                    tool_result = (
                        self.extract_mcp_result(
                            result
                        )
                    )

                    print(
                        "✅ MCP tool executed"
                    )

                except Exception as e:

                    tool_result = (
                        "MCP tool execution "
                        f"failed: {str(e)}"
                    )

                    print(
                        "❌ MCP tool failed:"
                    )

                    print(e)


            # ----------------------------------------------
            # Give MCP result back to LLM
            # ----------------------------------------------

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": tool_result,
                }
            )


        # ====================================================
        # SECOND LLM CALL
        # ====================================================

        print(
            "\n🤖 Sending MCP result "
            "back to LLM..."
        )

        final_response = self.call_llm(
            messages,
            include_tools=False,
        )

        final_message = (
            final_response
            .choices[0]
            .message
        )

        final_answer = (
            final_message.content
            or ""
        )


        # ====================================================
        # RETURN FINAL ANSWER
        # ====================================================

        return ResponsesAgentResponse(
            output=[
                self.create_text_output_item(
                    text=final_answer,
                    id=uuid.uuid4().hex,
                )
            ],
            custom_outputs=(
                request.custom_inputs
            ),
        )


# ============================================================
# 9. CREATE AGENT
# ============================================================

agent = GitHubMCPAgent()

print(
    "\n========================================"
)

print(
    "🚀 GitHub MCP ResponsesAgent READY"
)

print(
    "========================================"
)

In [0]:
request = ResponsesAgentRequest(
    input=[
        {
            "role": "user",
            "content": "Who am I on GitHub?"
        }
    ]
)

response = agent.predict(request)

print("\n================================")
print("FINAL AGENT RESPONSE")
print("================================")

for item in response.output:
    print(item)

In [0]:
for tool in mcp_tools:
    if "repo" in tool.name.lower():
        print("Tool:", tool.name)
        print("Description:", tool.description)
        print("Input schema:", tool.input_schema)
        print("-" * 80)

In [0]:
import os

path = "/Workspace/Users/sonadiggibyte@gmail.com/agent_github.py"

print("Exists:", os.path.exists(path))
print("Is file:", os.path.isfile(path))

In [0]:
import os

print(os.listdir("/Workspace/Users/sonadiggibyte@gmail.com"))

In [0]:
import importlib.util

file_path = "/Workspace/Users/sonadiggibyte@gmail.com/agent_github.py"

spec = importlib.util.spec_from_file_location(
    "agent_github",
    file_path
)

agent_module = importlib.util.module_from_spec(spec)

spec.loader.exec_module(agent_module)

GitHubMCPAgent = agent_module.GitHubMCPAgent

print("agent_github.py loaded successfully ✅")
print(GitHubMCPAgent)

In [0]:
agent = GitHubMCPAgent()

print("GitHub MCP Agent created successfully ✅")

In [0]:
# Inspect the actual MCP Tool object

tool = mcp_client.list_tools()[0]

print("Tool type:")
print(type(tool))

print("\nTool attributes:")
print(dir(tool))

In [0]:
import importlib.util

file_path = "/Workspace/Users/sonadiggibyte@gmail.com/agent_github.py"

spec = importlib.util.spec_from_file_location(
    "agent_github",
    file_path
)

agent_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(agent_module)

GitHubMCPAgent = agent_module.GitHubMCPAgent

print("agent_github.py loaded successfully ✅")

In [0]:
agent = GitHubMCPAgent()

print("Agent created successfully ✅")

TEST THE AGENT.PY HERE AND THEN LOG THAT IN MLFLOW

In [0]:
import mlflow

mlflow.set_experiment("/Shared/GitHub-MCP-Agent")

print("MLflow experiment set ✅")

In [0]:
with mlflow.start_run() as run:

    model_info = mlflow.pyfunc.log_model(
        artifact_path="github_mcp_agent",
        python_model=agent,
        code_paths=[
            "/Workspace/Users/sonadiggibyte@gmail.com/agent.py"
        ],
        extra_pip_requirements=[
            "mcp>=1.9",
            "databricks-mcp",
            "databricks-sdk[openai]",
            "databricks-openai",
            "mlflow>=3.16.1",
            "databricks-agents>=1.0.0"
        ]
    )

    print("Run ID:", run.info.run_id)
    print("Model URI:", model_info.model_uri)

In [0]:
import mlflow

mlflow.set_experiment(
    "/Shared/GitHub-MCP-Agent"
)

with mlflow.start_run() as run:

    model_info = mlflow.pyfunc.log_model(
        name="github_mcp_agent",

        python_model="/Workspace/Users/sonadiggibyte@gmail.com/agent_github.py",

        extra_pip_requirements=[
            "mcp>=1.9",
            "databricks-mcp",
            "databricks-sdk[openai]",
            "databricks-openai",
            "mlflow>=3.16.1",
            "databricks-agents>=1.0.0",
        ],
    )

    print("MLflow logging successful ✅")
    print("Run ID:", run.info.run_id)
    print("Model URI:", model_info.model_uri)

In [0]:
%pip install -U "mcp>=1.19,<2" "databricks-openai==0.17.1"

In [0]:
dbutils.library.restartPython()

In [0]:
import mcp
import databricks_openai
import mlflow

print("MCP:", mcp.__file__)
print("Databricks OpenAI:", databricks_openai.__file__)
print("MLflow:", mlflow.__version__)

In [0]:
import mlflow

mlflow.set_experiment("/Shared/GitHub-MCP-Agent")

with mlflow.start_run() as run:

    model_info = mlflow.pyfunc.log_model(
        name="github_mcp_agent",

        python_model="/Workspace/Users/sonadiggibyte@gmail.com/agent_github.py",

        extra_pip_requirements=[
            "mcp>=1.19,<2",
            "databricks-mcp",
            "databricks-sdk[openai]",
            "databricks-openai==0.17.1",
            "mlflow>=3.16.1",
            "databricks-agents>=1.0.0",
        ],
    )

    print("MLflow logging successful ✅")
    print("Run ID:", run.info.run_id)
    print("Model URI:", model_info.model_uri)

In [0]:
python_model="/Workspace/Users/sonadiggibyte@gmail.com/agent_github.py"

In [0]:
import mlflow

mlflow.set_experiment("/Shared/GitHub-MCP-Agent")

with mlflow.start_run() as run:

    model_info = mlflow.pyfunc.log_model(
        name="github_mcp_agent",

        python_model="/Workspace/Users/sonadiggibyte@gmail.com/agent_github.py",

        extra_pip_requirements=[
            "mcp>=1.19,<2",
            "databricks-mcp",
            "databricks-sdk[openai]",
            "databricks-openai==0.17.1",
            "mlflow>=3.16.1",
            "databricks-agents>=1.0.0",
        ],
    )

    print("MLflow logging successful ✅")
    print("Run ID:", run.info.run_id)
    print("Model URI:", model_info.model_uri)

In [0]:
registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name="workspace.default.github_mcp_agent"
)

print("Model registered ✅")
print("Name:", registered_model.name)
print("Version:", registered_model.version)

Deploy the registered agent to Model Serving.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

w = WorkspaceClient()

endpoint_name = "github-mcp-agent"

config = EndpointCoreConfigInput(name='default_config',
    served_entities=[
        ServedEntityInput(
            name="github_mcp_agent_v1",
            entity_name="workspace.default.github_mcp_agent",
            entity_version="1",
            workload_size="Small",
            scale_to_zero_enabled=True,
        )
    ]
)

endpoint = w.serving_endpoints.create(
    name=endpoint_name,
    config=config
)

print("Serving endpoint creation started ✅")
print("Endpoint:", endpoint_name)

In [0]:
status = w.serving_endpoints.get("github-mcp-agent")

print("Ready:", status.state.ready)
print("Config update:", status.state.config_update)

In [0]:
import time

time.sleep(30)

status = w.serving_endpoints.get("github-mcp-agent")

print("Ready:", status.state.ready)
print("Config update:", status.state.config_update)